# Graph2Edits 教程 3：模型展示

## 概述

本 notebook 解释 Graph2Edits 的模型主线：为什么它不是直接生成 reactants，而是对当前产物图做 **edit scoring**，并在自回归过程中一步一步把产物图改写成合成前体。

我们会重点对照 4 段源码：

1. `get_batch_graphs()`：图输入张量化。
2. `MPNEncoder.forward()`：图神经网络编码。
3. `Graph2Edits.compute_edit_scores()`：把节点 / 边 / 图表示转成 edit logits。
4. `predict()` 与 `BeamSearch`：解释自回归解码与 beam search。

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 读取教学版预处理产物 |
| 2 | 建立模型配置并实例化 Graph2Edits |
| 3 | 查看图张量与编码器输出 |
| 4 | 拆解单步 edit logits 的结构 |
| 5 | 用 oracle label 演示自回归 edit 解码 |
| 6 | 用 toy path 说明 beam search 保留机制 |

> 说明
>
> 这里不追求得到化学上正确的预测结果，因为我们没有加载训练好的 checkpoint。教学重点是“张量如何流动、edit 如何被解码、路径如何被搜索”。

## 源码对应

- 文件：`source_repos/Graph2Edits/utils/collate_fn.py`
- 文件：`source_repos/Graph2Edits/models/encoder.py`
- 文件：`source_repos/Graph2Edits/models/graph2edits.py`
- 文件：`source_repos/Graph2Edits/models/beam_search.py`
- 文件：`source_repos/Graph2Edits/train.py`

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
import torch
from rdkit import Chem


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "source_repos").exists() and (candidate / "teaching_demos").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录，请从 Chemical_Synthesis 项目内启动 notebook。")


PROJECT_ROOT = find_project_root(Path.cwd())
TUTORIAL_REL = Path("teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits")
REPO_REL = Path("source_repos/Graph2Edits")
TUTORIAL_ROOT = PROJECT_ROOT / TUTORIAL_REL
REPO_ROOT = PROJECT_ROOT / REPO_REL
PROCESSED_TRAIN_ROOT = TUTORIAL_ROOT / "data/processed/uspto_50k_demo/train"
BATCH_FILE = PROCESSED_TRAIN_ROOT / "without_rxn_class/batch-0.pt"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.beam_search import BeamSearch
from models.graph2edits import Graph2Edits
from prepare_data import apply_edit_to_mol
from train import build_model_config
from utils.reaction_actions import Termination
from utils.rxn_graphs import Vocab


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


required_files = [
    PROCESSED_TRAIN_ROOT / "train.file.kekulized",
    PROCESSED_TRAIN_ROOT / "bond_vocab.txt",
    PROCESSED_TRAIN_ROOT / "atom_lg_vocab.txt",
    BATCH_FILE,
]
for path in required_files:
    print(path.relative_to(PROJECT_ROOT), path.exists())

teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/train.file.kekulized True
teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/bond_vocab.txt True
teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/atom_lg_vocab.txt True
teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/without_rxn_class/batch-0.pt True


## Step 1: 读取教学版预处理产物

### 为什么需要这一步？

模型 notebook 不再从原始反应重新开始，而是直接接住上一份 notebook 产出的最小数据包：`ReactionData`、词表和 `batch-0.pt`。这样可以把注意力集中到“模型如何消费这些张量”。

### 计算主线总览

```text
product graph
    -> MolGraph / get_batch_graphs
    -> MPNEncoder
    -> optional Global Attention
    -> atom head / bond head / graph head
    -> flat edit logits
    -> decode edit
    -> apply_edit_to_mol
    -> next product graph
```

### 源码对应

- 文件：`source_repos/Graph2Edits/utils/collate_fn.py`
- 文件：`source_repos/Graph2Edits/models/encoder.py`
- 文件：`source_repos/Graph2Edits/models/graph2edits.py`

In [2]:
reaction_objects = joblib.load(PROCESSED_TRAIN_ROOT / "train.file.kekulized")
bond_vocab = Vocab(joblib.load(PROCESSED_TRAIN_ROOT / "bond_vocab.txt"))
atom_vocab = Vocab(joblib.load(PROCESSED_TRAIN_ROOT / "atom_lg_vocab.txt"))
graph_seq_tensors, edit_seq_labels, seq_mask = torch_load_compat(BATCH_FILE)

print("reaction count:", len(reaction_objects))
print("bond vocab size:", len(bond_vocab))
print("atom_lg vocab size:", len(atom_vocab))
print("sequence mask shape:", tuple(seq_mask.shape))

reaction count: 3
bond vocab size: 2
atom_lg vocab size: 3
sequence mask shape: (3, 3)


## Step 2: 建立模型配置并实例化 Graph2Edits

### 为什么需要这一步？

`Graph2Edits` 顶层模型本身并不复杂，但它依赖两个信息：

1. 输入特征维度与消息传递配置。
2. `bond_vocab` 与 `atom_lg_vocab` 的大小。

### 教学版与原版的差异

- 原版训练默认 `mpn_size=256`、`depth=10`。
- 教学版为了让 notebook 更快执行，会保留相同结构，但把隐藏维度和深度适当缩小。
- 这不会改变模型主线，只会降低计算量。

In [3]:
repo_args = {
    "use_rxn_class": False,
    "atom_message": False,
    "use_attn": False,
    "n_heads": 8,
    "mpn_size": 256,
    "depth": 10,
    "dropout_mpn": 0.15,
    "mlp_size": 512,
    "dropout_mlp": 0.2,
}

demo_args = {
    **repo_args,
    "mpn_size": 64,
    "depth": 3,
    "mlp_size": 128,
}

config_df = pd.DataFrame(
    [
        {"config": key, "repo_default": repo_args[key], "teaching_demo": demo_args[key]}
        for key in demo_args
    ]
)
display(config_df)

demo_config = build_model_config(demo_args)
model = Graph2Edits(config=demo_config, atom_vocab=atom_vocab, bond_vocab=bond_vocab, device="cpu")
model.eval()

param_count_m = sum(param.numel() for param in model.parameters()) / 1e6
print("parameter count (million):", round(param_count_m, 4))

,config,repo_default,teaching_demo
0,use_rxn_class,False,False
1,atom_message,False,False
2,use_attn,False,False
3,n_heads,8,8
4,mpn_size,256,64
5,depth,10,3
6,dropout_mpn,0.15,0.15
7,mlp_size,512,128
8,dropout_mlp,0.2,0.2


parameter count (million): 0.0829


## Step 3: 查看图张量与编码器输出

### 为什么需要这一步？

很多读者第一次看 Graph2Edits 会卡在这里：模型到底吃进去什么？`get_batch_graphs()` 返回的 6 个张量分别是什么？

这一节先不讨论 logits，只先看输入图张量和 `MPNEncoder` 的输出原子隐状态。

In [4]:
first_step_tensors, first_step_scopes = graph_seq_tensors[0]
f_atoms, f_bonds, a2b, b2a, b2revb, undirected_b2a = first_step_tensors

print("f_atoms:", tuple(f_atoms.shape))
print("f_bonds:", tuple(f_bonds.shape))
print("a2b:", tuple(a2b.shape))
print("b2a:", tuple(b2a.shape))
print("b2revb:", tuple(b2revb.shape))
print("undirected_b2a:", tuple(undirected_b2a.shape))
print("atom_scope:", first_step_scopes[0])
print("bond_scope:", first_step_scopes[1])

with torch.no_grad():
    encoder_output = model.encoder(tuple(t.to(model.device) for t in first_step_tensors), mask=None)

print("encoder output:", tuple(encoder_output.shape))
print("first two atom hidden vectors (first 8 dims):")
print(encoder_output[:2, :8])

f_atoms: (85, 85)
f_bonds: (181, 97)
a2b: (85, 4)
b2a: (181,)
b2revb: (181,)
undirected_b2a: (91, 2)
atom_scope: [(1, 27), (28, 30), (58, 27)]
bond_scope: [(1, 27), (28, 33), (61, 30)]
encoder output: (85, 64)
first two atom hidden vectors (first 8 dims):
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0940, 0.1237, 0.0000, 0.2057, 0.2273, 0.0597, 0.0000, 0.0000]])


## Step 4: 拆解单步 edit logits

### 为什么需要这一步？

Graph2Edits 的关键设计不是“直接给出一个 edit 类别”，而是把当前分子图里的所有可编辑位置拼成一个扁平向量：

`bond logits + atom logits + graph stop logit`

只有理解这个 layout，后面才能看懂 `predict()` 和 `BeamSearch.get_edit_from_logits()` 是怎么把一个 `argmax` 还原回具体 edit 的。

In [5]:
with torch.no_grad():
    edit_scores, prev_atom_hiddens, prev_atom_scope = model.compute_edit_scores(
        first_step_tensors,
        first_step_scopes,
    )

flat_scores = edit_scores[0]
product_mol = Chem.MolFromSmiles(reaction_objects[0].rxn_smi.split(">>")[-1])
bond_block = product_mol.GetNumBonds() * len(bond_vocab)
atom_block = product_mol.GetNumAtoms() * len(atom_vocab)

print("flat edit score shape:", tuple(flat_scores.shape))
print("bond block size:", bond_block)
print("atom block size:", atom_block)
print("stop block size: 1")

rows = []
topk = torch.topk(flat_scores, k=min(8, flat_scores.numel()))
for score, flat_idx in zip(topk.values.tolist(), topk.indices.tolist()):
    if flat_idx == flat_scores.numel() - 1:
        segment = "stop"
    elif flat_idx < bond_block:
        segment = "bond"
    else:
        segment = "atom"
    rows.append({"flat_index": int(flat_idx), "score": float(score), "segment": segment})

display(pd.DataFrame(rows))

flat edit score shape: (136,)
bond block size: 54
atom block size: 81
stop block size: 1


,flat_index,score,segment
0,111,0.066221,atom
1,99,0.062390,atom
2,93,0.057311,atom
3,84,0.056562,atom
4,129,0.050934,atom
5,69,0.050857,atom
6,102,0.048421,atom
7,96,0.047573,atom


## Step 5: 用 oracle label 演示自回归 edit 解码

### 为什么用 oracle？

如果直接让一个随机初始化模型去 `predict()`，看到的只是无意义的随机 edit。教学上更清晰的办法是：

- 继续使用真实数据处理阶段得到的 one-hot label；
- 把这个 label 当作“oracle logits”；
- 用与 `predict()` / `BeamSearch` 相同的解码规则，把它还原成具体 edit。

这样就能专注观察“扁平 index 如何回到 edit 语义”以及“一个 edit 如何把当前产物图推进到下一个状态”。

In [6]:
def normalize_mol(mol):
    return Chem.MolFromSmiles(Chem.MolToSmiles(mol))


def decode_edit_from_flat_vector(mol, flat_vector, bond_vocab, atom_vocab):
    flat_index = int(torch.argmax(flat_vector).item())
    max_bond_idx = mol.GetNumBonds() * len(bond_vocab)

    if flat_index == flat_vector.numel() - 1:
        return "Terminate", None, flat_index

    if flat_index < max_bond_idx:
        bond_logits = flat_vector[:max_bond_idx].reshape(mol.GetNumBonds(), len(bond_vocab))
        bond_idx, edit_idx = torch.nonzero(bond_logits == flat_vector[flat_index], as_tuple=True)
        bond_idx = int(bond_idx[-1].item())
        edit_idx = int(edit_idx[-1].item())
        bond = mol.GetBondWithIdx(bond_idx)
        edit_atom = sorted([bond.GetBeginAtom().GetAtomMapNum(), bond.GetEndAtom().GetAtomMapNum()])
        return bond_vocab.get_elem(edit_idx), edit_atom, flat_index

    atom_logits = flat_vector[max_bond_idx:-1].reshape(mol.GetNumAtoms(), len(atom_vocab))
    atom_idx, edit_idx = torch.nonzero(atom_logits == flat_vector[flat_index], as_tuple=True)
    atom_idx = int(atom_idx[-1].item())
    edit_idx = int(edit_idx[-1].item())
    atom = mol.GetAtomWithIdx(atom_idx)
    return atom_vocab.get_elem(edit_idx), atom.GetAtomMapNum(), flat_index


sample_idx = 1
rxn_data = reaction_objects[sample_idx]
current_mol = Chem.MolFromSmiles(rxn_data.rxn_smi.split(">>")[-1])
Chem.Kekulize(current_mol)
current_mol = normalize_mol(current_mol)

oracle_rows = []
for step_idx, step_labels in enumerate(edit_seq_labels):
    oracle_label = step_labels[sample_idx].float()
    edit, edit_atom, flat_index = decode_edit_from_flat_vector(current_mol, oracle_label, bond_vocab, atom_vocab)
    before_smi = Chem.MolToSmiles(current_mol)

    if edit == "Terminate":
        after_mol = normalize_mol(Termination(action_vocab="Terminate").apply(Chem.Mol(current_mol)))
        after_smi = Chem.MolToSmiles(after_mol)
    else:
        after_mol = normalize_mol(apply_edit_to_mol(Chem.Mol(current_mol), edit, edit_atom))
        after_smi = Chem.MolToSmiles(after_mol)
        current_mol = Chem.Mol(after_mol)

    oracle_rows.append(
        {
            "step": step_idx,
            "flat_index": flat_index,
            "decoded_edit": repr(edit),
            "edit_atom": repr(edit_atom),
            "before": before_smi,
            "after": after_smi,
        }
    )

display(pd.DataFrame(oracle_rows))
print("gold reactants:", rxn_data.rxn_smi.split(">>")[0])

,step,flat_index,decoded_edit,edit_atom,before,after
0,0,2,"('Delete Bond', (None, None))","[2, 3]",[O:1]=[C:2]([NH:3][c:4]1[cH:5][cH:6][cH:7][c:8...,[NH:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:1...
1,1,102,"('Attaching LG', '*O')",2,[NH:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:1...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...
2,2,159,'Terminate',None,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...


gold reactants: [NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O:18])[O-:19])[c:20]([S:21][c:22]2[c:23]([Cl:24])[cH:25][n:26][cH:27][c:28]2[Cl:29])[s:30]1)[OH:31]


## Step 6: 用 toy path 说明 beam search

### 为什么这里只做 toy path？

`BeamSearch.run_search()` 依赖真实模型给出概率分布。没有训练好的 checkpoint 时，真正跑出来的分数没有解释价值；但 beam search 的“路径乘法 + top-k 保留”机制仍然可以用一个极小的 toy 例子讲清楚。

### 核心思想

- 每条 path 记录累计概率 `prob`。
- 每一步先对单条 path 做 `top-k` 扩展。
- 所有扩展后的路径放到一起，再按累计概率只保留全局前 `beam_size` 条。

In [7]:
beam = BeamSearch(model=model, step_beam_size=3, beam_size=2, use_rxn_class=False)

toy_paths = [
    {"prob": 0.42, "edits": ["Delete Bond"]},
    {"prob": 0.18, "edits": ["Change Atom"]},
    {"prob": 0.31, "edits": ["Change Bond"]},
]

selected_paths = beam.get_top_k_paths(toy_paths)
print("all toy paths:")
display(pd.DataFrame(toy_paths))
print("\npaths kept when beam_size = 2:")
display(pd.DataFrame(selected_paths))

all toy paths:


,prob,edits
0,0.42,[Delete Bond]
1,0.18,[Change Atom]
2,0.31,[Change Bond]



paths kept when beam_size = 2:


,prob,edits
0,0.42,[Delete Bond]
1,0.31,[Change Bond]


## 小结

这份 notebook 把 Graph2Edits 的模型主线拆成了 4 个层次：

1. `get_batch_graphs()` 定义模型真正看到的图输入。
2. `MPNEncoder` 把图输入编码成原子隐状态。
3. `compute_edit_scores()` 把原子 / 键 / 图表示转成扁平 edit logits。
4. `predict()` 与 `BeamSearch` 则把单步 logits 包装成多步自回归编辑过程。

至此，Graph2Edits 的教学版主线已经完整闭环：环境、数据、模型三部分都可以和原仓库代码逐段对照。